# Load the required libraries

In [1]:
library(ggplot2)
library(patchwork)
library(scales)
library(tidyverse)
library(ggpubr)
library(ggbeeswarm)
library(ComplexHeatmap)
library(circlize)
library(survminer)
library(survival)
library(showtext)
library(ggbreak)
font_add("Arial", "/System/Library/Fonts/Supplemental/Arial.ttf")
showtext_auto()

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ lubridate 1.9.5     ✔ tibble    3.3.1
✔ purrr     1.2.2     ✔ tidyr     1.3.2
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ readr::col_factor() masks scales::col_factor()
✖ purrr::discard()    masks scales::discard()
✖ dplyr::filter()     masks stats::filter()
✖ dplyr::lag()        masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: grid

ComplexHeatmap version 2.26.1
Bioconductor page: http://bioconductor.org/packages/ComplexHeatmap/
Github page: https://github.com/jokergoo/ComplexHeatmap
Documentation: http://jokergoo.github.io/ComplexHeatmap-reference

If you use it in published research, please cite either one:
- Gu, Z. Complex Heatmap Visualization. iMeta 2022.
- Gu, Z. Complex heatmaps reve

# Paths

In [2]:
data_dir = "/Users/jawadalaaedeen/Desktop/PhD/NMC/results"
results_dir = "/Users/jawadalaaedeen/Desktop/PhD/NMC/results/poster_figures"

## Color palette

In [3]:
myeloid_palette <- c(
  "MDSCs" = "#332288",
  "M2 macrophages" = "#66A61E",
  "M1 macrophages" = "#00557F",
  "M1/M2 macrophages" = "#BBCCEE",
  "DCs" = "#E69F00",
  "Granulocytes" = "#B85AA6",
  "Mast cells" = "#8E006A"
)

lymphoid_palette <- c(
  "CD8+ T cells" = "#B2182B",
  "CD4+ T cells" = "#EF8A62",
  "Treg T cells" = "#FFCCCC",
  "NK cells" = "#0096FF",
  "B cells" = "#E6C84F",
  "PCs" = "#F4E58A"
)

non_immune_palette <- c(
  "Actin+ cells" = "#B39DDB",
  "Endothelial cells" = "#7F8F6A",
  "Lymphatic endothelial cells" = "#A6B08A",
  "Epithelial cells" = "#E3B07A"
)


other_palette <- c(
  "Tumor cells" = "#44AA99",
  "NFC" = "#DDDDDD"
)

full_palette <- c(
  myeloid_palette,
  lymphoid_palette,
  non_immune_palette,
  other_palette
)

tumor_site_colors <- c(
"Lung" = "#F0F0F0",     # very light
"Head and Neck" = "#B0B0B0",  # medium
"Extrahepatic" = "#505050" # darker
)


main_group_colors <- c(
  "Myeloid"  = "#332288",
  "Lymphoid" = "#B2182B",
  "Non-immune" = "#B39DDB",
  "Tumor" = "#44AA99",
  "NFC" = "#DDDDDD"
)

color_for_ratios <- c(
  "Granulocytes-T cells ratio"    = "#B85AA6",
  "Macrophages-T cells ratio" = "#66A61E",
  "MDSCs-T cells ratio" = "#332288"
)

# Loading the required dataset

In [4]:
celltype_metadata <- read_csv(
  paste0(data_dir, "/celltype_metadata_final.csv")
) %>%
  mutate(is_lung = ifelse(tumor_site == "Lung", "Lung", "Head and Neck/Extrahepatic")) %>%
  mutate(patient_num = as.numeric(str_extract(patient_exp, "(?<=NUT_)\\d+")),
         roi_num = as.numeric(str_extract(patient_exp, "(?<=ROI)\\d+"))) %>%
  arrange(patient_num, roi_num) %>%
  select(-patient_num, -roi_num)
celltype_metadata$patient_exp <- factor(celltype_metadata$patient_exp, levels = unique(celltype_metadata$patient_exp))
celltype_metadata$patient_ID <- factor(celltype_metadata$patient_ID, levels = unique(celltype_metadata$patient_ID))
patient_ID_order <- unique(celltype_metadata$patient_ID)
patient_exp_order <- unique(celltype_metadata$patient_exp)


myeloid <-  c("M1 macrophages", "M2 macrophages", "M1/M2 macrophages", "DCs", "Granulocytes", "MDSCs", "Mast cells")
lymphoid <- c("CD4+ T cells", "CD8+ T cells", "Treg T cells", "NK cells", "B cells", "PCs")
nonimmune <- c("Actin+ cells", "Endothelial cells", "Lymphatic endothelial cells", "Epithelial cells")

celltype_metadata <- celltype_metadata %>%
  mutate(
    cell_category_simplified = case_when(
      cell_category %in% myeloid ~ "Myeloid",
      cell_category %in% lymphoid ~ "Lymphoid",
      cell_category %in% nonimmune ~ "Non-immune",
      cell_category == "NFC" ~ "NFC",
      cell_category == "Tumor cells" ~ "Tumor"
    ),
    cell_category_simplified = factor(
      cell_category_simplified,
      levels = c("NFC", "Non-immune", "Lymphoid", "Myeloid", "Tumor")
    )
  ) 

Rows: 553331 Columns: 17
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (10): cell_type, cell_category, patient_ID, exp_name, patient_exp, tumor...
dbl  (7): cell_id, Cell Center X, Cell Center Y, run, rois, survival_time_mo...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [5]:
distances_scimap <- read_csv(file.path(data_dir, "distances_to_tumor_cells_scimap.csv"), show_col_types = FALSE) %>%
    mutate(cell_group = case_when(
        cell_category %in% myeloid ~ "Myeloid",
        cell_category %in% lymphoid ~ "Lymphoid",
        cell_category == "Tumor cells" ~ "Tumor",
        TRUE ~ "Non-immune"
    )) %>%
    filter(!cell_group %in% c("Tumor", "Non-immune"))    

In [6]:
survival_df <- celltype_metadata %>%
    select(patient_ID, tumor_site, survival_time_months) %>%
    distinct() %>%
    mutate(event = ifelse(patient_ID == "NUT_1", 0, 1),
        tumor_site = factor(tumor_site, levels = c("Lung", "Head and Neck", "Extrahepatic")), 
    patient_ID = factor(patient_ID, levels = patient_ID_order),
    is_lung = ifelse(tumor_site == "Lung", "Lung", "Head and Neck/Extrahepatic"),
        event = as.numeric(event),
        survival_time_months = as.numeric(survival_time_months)
      )

survival_df_order <- survival_df %>%
arrange(tumor_site, survival_time_months) %>%
mutate(patient_ID = factor(patient_ID, levels = unique(patient_ID)))

patient_ID_survival_order <-survival_df_order %>%
pull(patient_ID)

In [7]:
distances_scimap_cellcat <- distances_scimap %>%
    group_by(patient_exp, patient_ID, cell_category) %>%
    summarise(median_distance_um = median(`Tumor cells`, na.rm = TRUE)) %>%
    ungroup() %>%
    group_by(patient_ID, cell_category) %>%
    summarise(median_distance_um = median(median_distance_um, na.rm = TRUE)) %>%
    ungroup() %>%
    complete(patient_ID, cell_category) %>%
    left_join(., survival_df) 

`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by patient_exp, patient_ID, and
  cell_category.
ℹ Output is grouped by patient_exp and patient_ID.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(patient_exp, patient_ID, cell_category))` for
  per-operation grouping (`?dplyr::dplyr_by`) instead.
`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by patient_ID and cell_category.
ℹ Output is grouped by patient_ID.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(patient_ID, cell_category))` for per-operation
  grouping (`?dplyr::dplyr_by`) instead.
Joining with `by = join_by(patient_ID)`


In [8]:
distances <- read_csv(file.path("/Users/jawadalaaedeen/Desktop/PhD/NMC/results", "cell_distances_to_tumor_contour.csv"), show_col_types = FALSE) %>%
    mutate(cell_group = case_when(
        cell_category %in% myeloid ~ "Myeloid",
        cell_category %in% lymphoid ~ "Lymphoid",
        cell_category == "Tumor cells" ~ "Tumor",
        TRUE ~ "Non-immune"
    ))

    averaged_distances <- distances %>%
    # filter(tumor_signed_dist_to_contour_um >= 0) %>%
    group_by(patient_ID, patient_exp, cell_group) %>%
    summarise(avg_distance = median(tumor_signed_dist_to_contour_um)) %>%
    group_by(patient_ID, cell_group) %>%
    summarise(avg_distance = median(avg_distance)) %>%
    filter(!cell_group %in% c("Tumor", "Non-immune"))

averaged_distances_pivot <- averaged_distances %>%
    pivot_wider(names_from = cell_group, values_from = avg_distance)

plot_df <- averaged_distances_pivot %>%
  select(patient_ID, Myeloid, Lymphoid) %>%
  drop_na(Myeloid, Lymphoid) %>%
  pivot_longer(cols = c(Myeloid, Lymphoid),
               names_to = "cell_group",
               values_to = "median_distance_um")



`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by patient_ID, patient_exp, and cell_group.
ℹ Output is grouped by patient_ID and patient_exp.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(patient_ID, patient_exp, cell_group))` for
  per-operation grouping (`?dplyr::dplyr_by`) instead.
`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by patient_ID and cell_group.
ℹ Output is grouped by patient_ID.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(patient_ID, cell_group))` for per-operation grouping
  (`?dplyr::dplyr_by`) instead.


In [29]:
enrich_df <- read_csv(file.path(data_dir, "layer_enrichment_results.csv"), show_col_types = FALSE)
enrich_df

patient_ID,tumor_layer,cell_type,layer_proportion,global_proportion,fold_enrichment,log2_fold_enrichment,diff
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
NUT_1,stroma 0-25 µm,Actin+ cells,0.0871135793,0.0458263187,1.7393835,0.77016394,4.128726e-02
NUT_1,stroma 0-25 µm,B cells,0.0025460498,0.0025694590,0.9622332,-0.05681163,-2.340920e-05
NUT_1,stroma 0-25 µm,CD4+ T cells,0.0099471930,0.0055026332,1.8180396,0.85601624,4.444560e-03
NUT_1,stroma 0-25 µm,CD8+ T cells,0.0168690295,0.0121758193,1.3073331,0.37924157,4.693210e-03
NUT_1,stroma 0-25 µm,DCs,0.0001717033,0.0002534020,0.6914063,0.46760555,-8.169868e-05
NUT_1,stroma 0-25 µm,Endothelial cells,0.0578582960,0.0326263729,1.7965326,0.84488628,2.523192e-02
NUT_1,stroma 0-25 µm,Epithelial cells,0.0000000000,0.0000000000,NA,NA,0.000000e+00
NUT_1,stroma 0-25 µm,Granulocytes,0.0004894763,0.0003851657,0.7575135,0.59934343,1.043106e-04
NUT_1,stroma 0-25 µm,Lymphatic endothelial cells,0.0362072078,0.0205106406,2.1971828,1.09437839,1.569657e-02


In [ ]:
order_cats <- distances_scimap_cellcat %>%
  group_by(cell_category) %>%
  summarise(
    med = median(median_distance_um, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(med) %>%   # <-- decreasing order
  pull(cell_category)

distances_scimap_cellcat$cell_category <- factor(
  distances_scimap_cellcat$cell_category,
  levels = order_cats
)

[1] MDSCs             M2 macrophages    M1 macrophages    B cells          
 [5] Treg T cells      Granulocytes      CD8+ T cells      M1/M2 macrophages
 [9] NK cells          Mast cells        DCs               CD4+ T cells     
[13] PCs              
13 Levels: MDSCs M2 macrophages M1 macrophages B cells ... PCs

## Figures

In [19]:
myeloid_select = c("Granulocytes", "M2 macrophages", "MDSCs")

#calculate infiltration, so percentage of cells in each patient that are within the stroma_tumor, tumor_0_25, and tumor_25+ layers
non_contour_cells <- distances %>%
filter(!tumor_layer %in% c("stroma 0-25 µm", "tumor 0-25 µm", "tumor 25+ µm")) %>%
select(patient_exp, patient_ID, cell_category, tumor_total_area_um2, non_tumor_area_um2) %>%
group_by(patient_exp, patient_ID, cell_category, tumor_total_area_um2, non_tumor_area_um2) %>%
summarise(n_outside = n()) %>%
ungroup()

infiltration_df <- distances %>%
    filter(tumor_layer %in% c("stroma 0-25 µm", "tumor 0-25 µm", "tumor 25+ µm")) %>%
    group_by(patient_exp, patient_ID, cell_category) %>%
    summarise(n_infiltrating = n()) %>%
    ungroup() %>%
    left_join(
        non_contour_cells,
        by = c("patient_exp", "patient_ID", "cell_category")
        ) %>%
    mutate(n_infiltrating_normalized = (n_infiltrating / sqrt(tumor_total_area_um2)),
            n_outside_normalized = (n_outside / sqrt(non_tumor_area_um2))) %>%
    mutate(infiltration_ratio = log2(n_infiltrating_normalized / n_outside_normalized)) %>%
    select(!patient_exp) %>%
    group_by(patient_ID, cell_category) %>%
    summarise(median_infiltration = median(infiltration_ratio, na.rm = TRUE)) %>%
    mutate(patient_ID = str_replace(patient_ID, "NMC", "NUT")) %>%
    filter(cell_category %in% c(myeloid, lymphoid)) %>%
    #arrange patient_ID according to patient_ID_order
    ungroup() %>%
    filter(cell_category %in% c(myeloid_select, "CD8+ T cells", "CD4+ T cells")) %>%
    mutate(patient_ID = factor(patient_ID, levels = patient_ID_order),
    cell_category = factor(cell_category, levels = c("CD8+ T cells", "CD4+ T cells", myeloid_select))) %>%
    arrange(patient_ID, cell_category)


`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by patient_exp, patient_ID, cell_category,
  tumor_total_area_um2, and non_tumor_area_um2.
ℹ Output is grouped by patient_exp, patient_ID, cell_category, and
  tumor_total_area_um2.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(patient_exp, patient_ID, cell_category,
  tumor_total_area_um2, non_tumor_area_um2))` for per-operation grouping
  (`?dplyr::dplyr_by`) instead.
`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by patient_exp, patient_ID, and
  cell_category.
ℹ Output is grouped by patient_exp and patient_ID.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(patient_exp, patient_ID, cell_category))` for
  per-operation grouping (`?dplyr::dplyr_by`) instead.
`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by patient_ID and cell_category.
ℹ Output is grouped by patient_ID.


In [21]:
infiltration_mat <- infiltration_df %>%
  select(patient_ID, cell_category, median_infiltration) %>%
  pivot_wider(names_from = patient_ID, values_from = median_infiltration) %>%
  #Make cell_category like cat_order   
  mutate(cell_category = factor(cell_category, levels = order_cats)) %>%
  column_to_rownames(var = "cell_category") %>%
  as.matrix()

In [28]:
# ── Colors ────────────────────────────────────────────────────────────────────
col_fun <- colorRamp2(
  c(-5, 0, 5),
  c("#2166AC", "white", "#B2182B")  # nicer blue-white-red
)

#rowsplit
row_split <- ifelse(rownames(infiltration_mat) %in% myeloid_select, "Myeloid", "Lymphoid")
row_split <- factor(row_split, levels = c("Lymphoid", "Myeloid"))  # control order

#column split
column_split <- survival_df_order %>%
  select(patient_ID, tumor_site) %>%
  distinct() %>%
  pull(tumor_site) %>%
  factor(levels = c("Lung", "Head and Neck", "Extrahepatic"))


# Draw heatmap
ht <- ComplexHeatmap::Heatmap(
  infiltration_mat,
name = "Tumor Infiltration",
col = col_fun,
na_col = "grey85",
row_title = "Cell types",
column_title = c("Lung", "Head and Neck", "Extrahepatic"),
show_row_names = TRUE,
show_column_names = TRUE,
column_names_rot = 45,
cluster_rows = FALSE,
cluster_columns = FALSE,
#heatmap legend
heatmap_legend_param = list(
    title            = "Infiltration (log2 ratio)",
    title_gp         = gpar(fontsize = 13),
    labels_gp        = gpar(fontsize = 13),
    at               = c(-5, 0, 5),
    legend_direction = "vertical"
  ),
row_split = row_split,
column_split = column_split,
column_names_gp = gpar(fontsize = 13),
row_names_gp = gpar(fontsize = 13),
column_title_gp = gpar(fontsize = 13),
row_title_gp = gpar(fontsize = 13),
row_gap           = unit(3, "mm"),
column_gap        = unit(7, "mm"),
row_names_side = "left",
rect_gp = gpar(col = "white", lwd = 0.5), 
)

#save
pdf(
  file   = paste0(results_dir, "/infiltration_heatmap.pdf"),
  width  = 11,
  height = 4
)
draw(ht,
     heatmap_legend_side   = "right",
     annotation_legend_side = "right",
     padding = unit(c(5, 5, 5, 5), "mm"))
dev.off()

agg_record_90e23ebc5d3e 
                      2